# Prompt-based LLM coding of the survey dataset

Supplementary analysis for *A Framework for Deductive Semantic Content Analysis at Scale in Science Education Using Text Embeddings* (Mjaaland, Kreutzer, Tyseng, Fussell, Passante, Holmes, Malthe-Sørenssen & Odden).

This notebook compares Deductive Semantic Content Analysis (DeSCA, see [DeSCA.ipynb](DeSCA.ipynb)) with an alternative approach that is increasingly used for qualitative coding: prompting a chat-based large language model (LLM) to assign each response to a category. Instead of defining each category through a handful of example responses (as DeSCA does), the LLM is given a written codebook and asked to classify one response at a time.

**Set-up of the comparison**

* **Data:** the open-ended responses about sources of variability in the experimental scenarios (Holmes et al., 2025; [Zenodo](https://doi.org/10.5281/zenodo.16912394)), with the audited human codes used in the paper.
* **Task:** the *simulated exhaustive coding task* from the paper. Responses coded as *Other* are excluded, and every remaining response is assigned to exactly one of Limitations (L), Physical Principles (P), or Statistics (S). This leaves 2,484 responses.
* **Model:** OpenAI `gpt-4.1-mini-2025-04-14`, accessed through the OpenAI API, one request per response, with JSON output (category, confidence, evidence, short justification).
* **Prompts:** two variants of the codebook:
  * a **short prompt** with a one-sentence definition of each category, and
  * a **long prompt** with fuller definitions and explicit inclusion/exclusion criteria for each category.
* **Evaluation:** the same agreement statistics as in the paper (Cohen's κ, weighted F1, and multiclass MCC), computed against the human codes, plus confusion matrices.

Classifying all 2,484 responses takes roughly two hours per prompt through the API, so the full classifications are provided as data files in [data/llm_coding/](data/llm_coding/) and loaded in Step 5. Steps 3 and 4 show how they were produced and can be re-run with your own API key on a small subset (or the full dataset).

*This notebook is adapted from a notebook by Ben Zwickl (RIT, UiO) and Tor Ole B. Odden (UiO) prepared for the PERC 2026 workshop "Learn to Analyze Open-Ended Survey Data with Text Embeddings".*

## Step 0: Setup

The analysis of the precomputed results only needs `pandas` and `scikit-learn`. Re-running the classifications additionally needs the `openai` package and an OpenAI API key.

The API key is read from the environment variable `OPENAI_API_KEY`, or requested interactively. **Never write an API key into the notebook.** Set `RUN_LLM = True` to call the API; with the default `False`, the notebook only evaluates the precomputed classifications.

Other providers can be swapped in with small changes to `classify_response()`, e.g. locally hosted models through [Ollama](https://ollama.com/) (which avoids sending human-subjects data to a third party), the [Gemini API](https://ai.google.dev/gemini-api/docs), the [Claude API](https://platform.claude.com/docs/en/api/overview), or models on [Hugging Face](https://huggingface.co/).

In [1]:
# Set to True to call the OpenAI API (requires an API key and incurs a small cost)
RUN_LLM = False

# Set to True to re-classify the full dataset (~2 hours and ~$0.50 per prompt with gpt-4.1-mini)
RUN_FULL_DATASET = False

MODEL = "gpt-4.1-mini-2025-04-14"

In [2]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display
from sklearn.metrics import (classification_report, cohen_kappa_score,
                             confusion_matrix, f1_score, matthews_corrcoef)

client = None
if RUN_LLM:
    # !pip install -q openai
    from getpass import getpass
    from openai import OpenAI

    api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
    client = OpenAI(api_key=api_key)

## Step 1: Load the data

Three data files are used, all in [data/llm_coding/](data/llm_coding/) (copies of the files in the [PERC 2026 workshop repository](https://github.com/Zwicklab/PERC2026Workshop)):

1. `responses_unclassified_subset.csv`: 30 responses (10 per category) used to demonstrate the classification loop.
2. `classified_student_responses_short_prompt.csv`: all 2,484 L/P/S responses classified with the short prompt.
3. `classified_student_responses_long_prompt.csv`: the same responses classified with the long prompt.

In each file, `code` is the human code and (in files 2 and 3) `code_AI`, `confidence` and `reason` are the LLM outputs.

In [3]:
DATA_DIR = Path("data/llm_coding")
REMOTE = "https://github.com/Zwicklab/PERC2026Workshop/raw/refs/heads/main/"

def load_csv(name):
    """Load from the local data folder, falling back to the workshop repository."""
    local = DATA_DIR / name
    return pd.read_csv(local if local.exists() else REMOTE + name)

df_sub = load_csv("responses_unclassified_subset.csv")
df_sub.head()

,Unnamed: 0,response,Experiment,code
0,614,Uncertainty in ruler,BM,L
1,1315,Force exerted by the students upon release.,PM,L
2,2765,other interfereing manetic fields,SG,L
3,1240,There has to be one correct answer which is wh...,PM,L
4,1797,Human error eg. is the table accidentally shak...,PM,L


## Step 2: Write the prompts

Each prompt contains (a) general instructions, (b) the category definitions, (c) the response to classify, and (d) the required output format. The category definitions follow the coding scheme of Stump, Dew, et al. (2023) used throughout the paper.

The **short prompt** gives a one-sentence description of each category.

In [4]:
# Short prompt with 1 sentence descriptions of each code

prompt_short = """
You are a classifier for survey responses that were from an introductory
university-level physics course where students are writing short explanations
about measurement uncertainty in different experimental scenarios.

Classify students' responses into exactly one of the following categories.

List of reasoning categories:
Category 1
Category Name: Limitations
Category Code: L
Category Description: Reasoning that uncertainty arises from an inability to perfectly measure and model all aspects of a real-world experiment.
------------
Category 2
Category Name: Physical Principles
Category Code: P
Category Description: Reasoning that uncertainty is inherent in the theoretical abstraction of an experiment.
------------
Category 3
Category Name: Statistics
Category Code: S
Category Description: Reasoning that uncertainty is inherent to the nature of experiments and repeating trials.
------------

Rules:
- Choose the best matching reasoning category.
- Return only valid JSON.
- Do not invent a new category.

Student response:
{student_response}

Return JSON only with this structure:
{{
  "selected_Category_Name": "...",
  "selected_Category_Code": "...",
  "confidence": 0.0,
  "evidence_from_response": ["...", "..."],
  "reasoning_summary": "brief explanation"
}}
"""

The **long prompt** adds fuller descriptions and explicit inclusion and exclusion criteria for each category.

In [5]:
# Long prompt: codebook definitions with inclusion/exclusion criteria

prompt_long = """
You are a classifier for survey responses that were from an introductory
university-level physics course where students are writing short explanations
about measurement uncertainty in different experimental scenarios.

Classify students' responses into exactly one of the following categories.

List of reasoning categories:
Category 1
Category Name: Limitations
Category Code: L
Category Description: A distribution in measured values is caused by practical limitations in an experiment owing to our inability to perfectly model and measure a real-world system.

Include the following for the Limitations category:
* Human error
* Imperfections in apparatus
* Imperfections in implementation of experiment
* Generic statements about equipment generally (e.g. “environmental influences”, “minor differences in equipment”)

Exclude the following for the Limitations category:
* Vaguely worded answers about “uncertainty” because it could be principles.
* “Random errors” also excluded because it could be principles or statistics. Unclear what the randomness is due to.
* Physical mechanisms that determine the position of the central value but are not varying between experimental trials, e.g. gravity in PM.
* “Error?” “experimental error”, “Measurement uncertainty” or “measurement error” because they are too abstract and vague.
------------
Category 2
Category Name: Physical Principles
Category Code: P
Category Description: A distribution in measured values is caused by some principle of theoretical physics (theoretical abstraction of an experiment) or of experimentation (measurement is inherently random).

Include the following for the Physical Principles category:
* General descriptions of randomness in relevant contexts (e.g. Brownian motion)
* General descriptions of quantum mechanical effects
* Statements about measurement uncertainty being inherent, inevitable, unavoidable, etc. Same for “inherent uncertainty” or other statements that uncertainty/error will always be present in an experiment.
* Any discussion of statistical error/statistical properties (as long as it’s unambiguously not referring to a physical principle). Can include generic statistics “jargon” (e.g., normal distributions, Gaussians, standard deviation).
* Discussion of statistics/statistical error, including references to specific distributions (e.g. Gaussian, normal, etc.).

Exclude the following for the Physical Principles category:
* “Random error” (we can’t be sure what students mean by the term error, as it is often used to imply mistakes).
* Any source that specifically discusses a practical limitation of the experiment, i.e. a specific physical mechanism by which measurement is inherently random/uncertain, e.g. precision of a ruler or random air currents.
* Physical mechanisms that determine the position of the central value but are not varying between experimental trials, e.g. gravity in PM.

------------
Category 3
Category Name: Statistics
Category Code: S
Category Description: Reasoning that uncertainty is inherent to the nature of experiments and repeating trials.

Include the following for the Statistics category:
* Text about what happens when more data are collected (e.g. more data makes the shape more narrow/won’t change anything, etc)
* Anything about the effect of outliers (more/fewer)
* Anything that is just about data

Excludes the following for the Statistics category:
* Talking about physics
* Just describing what is there (i.e. the shape of a histogram)
------------

Rules:
- Choose the best matching reasoning category.
- Return only valid JSON.
- Do not invent a new category.

Student response:
{student_response}

Return JSON only with this structure:
{{
  "selected_Category_Name": "...",
  "selected_Category_Code": "...",
  "confidence": 0.0,
  "evidence_from_response": ["...", "..."],
  "reasoning_summary": "brief explanation"
}}
"""

## Step 3: Classify a single response

`classify_response()` sends one prompt to the model and parses the returned JSON. The model sometimes wraps its JSON in Markdown code fences, which are stripped before parsing.

In [6]:
PROMPTS = {"short": prompt_short, "long": prompt_long}

def classify_response(response_text, prompt_name="long"):
    """Classify one survey response with the LLM and return the parsed JSON as a dict."""
    result = client.responses.create(
        model=MODEL,
        input=PROMPTS[prompt_name].format(student_response=response_text),
    )
    json_string = result.output_text.strip().removeprefix("```json").removesuffix("```")
    return json.loads(json_string)

if RUN_LLM:
    print(df_sub.loc[0, "response"])
    print(json.dumps(classify_response(df_sub.loc[0, "response"]), indent=2))
else:
    print("RUN_LLM is False: skipping API call.")

RUN_LLM is False: skipping API call.


## Step 4: Loop over responses

The same function is applied to every response, and the code, confidence and justification are appended to the table. By default this runs on the 30-response demonstration subset; set `RUN_FULL_DATASET = True` to reproduce the full classifications (the results are written to `data/llm_coding/rerun_*.csv` so the provided files are not overwritten).

In [7]:
def classify_dataframe(df, prompt_name, progress_every=50):
    """Classify every response in df and return a copy with code_AI, confidence and reason columns."""
    df = df.copy()
    for n, (i, row) in enumerate(df.iterrows()):
        parsed = classify_response(row["response"], prompt_name)
        df.loc[i, "code_AI"] = parsed["selected_Category_Code"]
        df.loc[i, "confidence"] = parsed["confidence"]
        df.loc[i, "reason"] = parsed["reasoning_summary"]
        if n % progress_every == 0:
            print(f"{prompt_name}: {n}/{len(df)}")
    return df

if RUN_LLM:
    df_sub_classified = classify_dataframe(df_sub, "long", progress_every=5)
    display(df_sub_classified[["response", "code", "code_AI", "confidence", "reason"]])

    if RUN_FULL_DATASET:
        base = load_csv("classified_student_responses_long_prompt.csv")
        base = base[["ResponseId", "response", "Experiment", "code"]]
        for name in PROMPTS:
            out = classify_dataframe(base, name)
            out.to_csv(DATA_DIR / f"rerun_classified_student_responses_{name}_prompt.csv", index=False)
else:
    print("RUN_LLM is False: skipping API calls.")

RUN_LLM is False: skipping API calls.


## Step 5: Evaluate the full classifications

We load the full LLM classifications for both prompts and compare them with the human codes using the same statistics as in the paper:

* **Cohen's κ:** chance-corrected agreement, treating the LLM as a second coder.
* **Weighted F1:** per-category harmonic mean of precision and recall, averaged with weights proportional to category size.
* **Matthews correlation coefficient (MCC):** a balanced measure of agreement across all categories that is less inflated by performance on the largest category.

Rows without an LLM code (if any) are excluded.

In [8]:
LABELS = ["L", "P", "S"]

def compute_metrics(y_true, y_pred):
    return {
        "kappa": cohen_kappa_score(y_true, y_pred),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }

results = {}
for name in ["short", "long"]:
    df = load_csv(f"classified_student_responses_{name}_prompt.csv")
    df = df[~df["code_AI"].isna()]
    results[name] = df

summary = pd.DataFrame(
    {f"{name.capitalize()} prompt": {"N": len(df), **compute_metrics(df["code"], df["code_AI"])}
     for name, df in results.items()}
).T
summary["N"] = summary["N"].astype(int)
summary.style.format({"kappa": "{:.2f}", "f1_weighted": "{:.2f}", "mcc": "{:.2f}"})

,N,kappa,f1_weighted,mcc
Short prompt,2484,0.67,0.93,0.70
Long prompt,2484,0.67,0.93,0.68


### Confusion matrices

Rows are the human codes and columns the LLM codes. The left table shows counts; the right table shows proportions within each human-coded category (i.e. recall along the diagonal).

In [9]:
def show_confusion(y_true, y_pred, title, labels=LABELS):
    confusion_counts = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=labels),
        index=pd.Index(labels, name="Actual code"),
        columns=pd.Index(labels, name="Predicted code"),
    )

    confusion_proportions = confusion_counts.div(confusion_counts.sum(axis=1), axis=0)

    table_styles = [
        {"selector": "caption", "props": [("caption-side", "top"), ("font-weight", "bold"), ("font-size", "1.05em")]},
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "center")]},
    ]

    count_style = (
        confusion_counts.style
        .set_caption(f"{title}: number of responses")
        .format("{:,d}")
        .background_gradient(cmap="Blues", axis=None)
        .set_table_styles(table_styles)
    )

    proportion_style = (
        confusion_proportions.style
        .set_caption(f"{title}: percentage within each human code")
        .format("{:.1%}")
        .background_gradient(cmap="Greens", axis=1, vmin=0, vmax=1)
        .set_table_styles(table_styles)
    )

    display(HTML(
        "<div style=\"display:flex; gap:24px; align-items:flex-start; flex-wrap:wrap;\">"
        + count_style.to_html()
        + proportion_style.to_html()
        + "</div>"
    ))

for name, df in results.items():
    show_confusion(df["code"], df["code_AI"], f"{name.capitalize()} prompt")

Predicted code,L,P,S
Actual code,,,
L,"2,035",34,142
P,8,123,25
S,0,0,117
Predicted code,L,P,S
Actual code,,,
L,92.0%,1.5%,6.4%
P,5.1%,78.8%,16.0%
S,0.0%,0.0%,100.0%


Predicted code,L,P,S
Actual code,,,
L,"2,089",72,50
P,8,147,1
S,0,68,49
Predicted code,L,P,S
Actual code,,,
L,94.5%,3.3%,2.3%
P,5.1%,94.2%,0.6%
S,0.0%,58.1%,41.9%


### Per-category performance

In [10]:
for name, df in results.items():
    print(f"=== {name.capitalize()} prompt ===")
    print(classification_report(df["code"], df["code_AI"], labels=LABELS, digits=2))

=== Short prompt ===
              precision    recall  f1-score   support

           L       1.00      0.92      0.96      2211
           P       0.78      0.79      0.79       156
           S       0.41      1.00      0.58       117

    accuracy                           0.92      2484
   macro avg       0.73      0.90      0.78      2484
weighted avg       0.96      0.92      0.93      2484

=== Long prompt ===
              precision    recall  f1-score   support

           L       1.00      0.94      0.97      2211
           P       0.51      0.94      0.66       156
           S       0.49      0.42      0.45       117

    accuracy                           0.92      2484
   macro avg       0.67      0.77      0.70      2484
weighted avg       0.94      0.92      0.93      2484



### Self-reported confidence

The model also reports a confidence score for each classification. If these scores were informative, misclassified responses should receive lower confidence than correctly classified ones.

In [11]:
conf = pd.DataFrame({
    f"{name.capitalize()} prompt": df.assign(correct=df["code"] == df["code_AI"])
                                     .groupby("correct")["confidence"].mean()
    for name, df in results.items()
}).rename(index={True: "Correct", False: "Incorrect"})
conf.round(2)

,Short prompt,Long prompt
correct,,
Incorrect,0.89,0.86
Correct,0.88,0.86


## Step 6: Comparison with DeSCA

For reference, the table below places the LLM results next to the DeSCA results on the same simulated exhaustive coding task (paper, Table 2 and Table S9: mean over 100 runs with resampled centroids, 28 example responses per run).

In [12]:
desca = pd.DataFrame(
    [
        ["DeSCA, lowest-scoring model (Jina small v2)", 0.95, 0.74, 0.74],
        ["DeSCA, mean across models", np.nan, 0.78, np.nan],
        ["DeSCA, highest-scoring model (Intfloat large instruct, STS)", 0.96, 0.83, 0.83],
    ],
    columns=["Method", "f1_weighted", "kappa", "mcc"],
).set_index("Method")

llm = summary.rename(index=lambda s: f"LLM ({MODEL}), {s.lower()}")[["f1_weighted", "kappa", "mcc"]]
pd.concat([desca, llm]).round(2)

,f1_weighted,kappa,mcc
"DeSCA, lowest-scoring model (Jina small v2)",0.95,0.74,0.74
"DeSCA, mean across models",NaN,0.78,NaN
"DeSCA, highest-scoring model (Intfloat large instruct, STS)",0.96,0.83,0.83
"LLM (gpt-4.1-mini-2025-04-14), short prompt",0.93,0.67,0.70
"LLM (gpt-4.1-mini-2025-04-14), long prompt",0.93,0.67,0.68


## Cost

Classifying all ~2,500 responses with one prompt cost approximately \$0.50 with `gpt-4.1-mini` (about \$0.0002 per short response) and took roughly two hours of sequential API calls.

Note that LLM outputs are not deterministic: re-running the classification can give somewhat different codes for some responses, and results depend on the exact model version. The provided data files record the classifications reported in the paper's supplementary information.